# Levantamento das features

## Features de Satisfação

### **Feature 1: SatisfactionScore**

**Justificativa**: 

Utilizar variáveis de satisfação para retornar um valor geral satisfatório entre o funcionário e a empresa. Cada uma mede um aspecto diferente, mas nenhuma delas separadamente diz que no geral o funcionário está satisfeito ou insatisfeito com a empresa. 

Para essa abordagem utilizaremos JobSatisfaction, EnvironmentSatisfaction e RelationshipSatisfaction. Temos a opção de adicionarmos WorkLifeBalance, porém essa variável mede o equilíbrio entre a vida pessoal e o trabalho, portanto o conceio pode ser parecido mas são conceitos diferentes.

Poderíamos adicionar uma feature a mais para medir o relacionamento do funcionário com a vida pessoal, porém, a correlação entre as duas seriam altas e teríamos uma multicolineariedade, caso as variáveis fossem selecionadas para modelagem. Tomamos a decisão de manter apenas as três variáveis que medem a satisfação do funcionário em relação a empresa.

- JobSatisfaction            ->  satisfação com o cargo
- EnvironmentSatisfaction    ->  satisfação com o ambiente de trabalho
- RelationshipSatisfaction   ->  satisfação com os relacionamentos (colegas/gestor)

**Calculo:** média de (JobSatisfaction + EnvironmentSatisfaction + RelationshipSatisfaction) / 3 


### **Feature 2: CriticalSatisfactionFlag**

**Justificativa**: Criar um indicador binário de satisfação crítica. Com isso facilita o modelo em separar os funcionários com risco maior de attrito. O threshold de 2 foi escolhido para identificar somente quem está insatisfeito (insatisfação baixa e média).

**Cálculo**: 1 se SatisfactionScore <= 2, senão 0


## Features de Carreira e Tempo

### **Feature 3: RatioCareerCompany**

**Justificativa**:  

Mede qual foi a proporção de anos trabalhados na empresa TechCorp Brasil em relação a quantidade total de anos trabalhados. Essa feature pode identificar funcionários que fizeram a sua carreira dentro da empresa TechCorp Brasil (mais difícil de entrar em attrition). A feature também pode encontrar funcionários que mudaram de emprego muitas vezes (mais fácil de entrar em attrition) e assim o modelo pode conseguir identificar padrões e realizar previsões mais próximas da realidade.

- Valor alto (ex: 0.90) = fez carreira aqui, mais estável
- Valor baixo (ex: 0.20) = passou por muitas empresas, mais fácil de sair da empresa
- Adicionamos +1 ao denominador para evitar divisão por zero

**Cálculo:** YearsAtCompany / (TotalWorkingYears + 1)

### **Feature 4: RoleStagnation**

**Justificativa**:
Mede o tempo que o funcionário está no mesmo cargo em relação ao tempo total que ele tem trabalhando na empresa. Essa feature consegue identificar os funcipnários que possuem muitos anos em uma mesma função. Sabemos que um funcionário que fica muitos anos em um mesmo cargo, pode gerar insatisfação e não contentamento com o seu trabalho e pode acabar buscando por outras oportunidades fora da empresa.

- valor próximo de 1.0 = nunca mudou de cargo
- valor baixo = mudou de cargo recentemente

Cálculo: YearsInCurrentRole / (YearsAtCompany + 1)


### **Feature 5: YearsWithoutPromotion**

**Justificativa**:
Mede quanto tempo o funcionário está sem promoção em relação ao tempo total de anos trabalhados na empresa. Ao realizar o levantamento dessa feature, surgiu a dúvida de estar fortemente correlacionada com a **Feature 2 RoleStagnation**, porém, são medidas diferentes. Aqui estamos medindo a promoção de um nível atual para um nível acima, por exemplo: Um funcionário que está a muito tempo como Pleno (mid-level), tende a estar insatisfeito com a situação atual e pode procurar oportunidades em um nível acima em outra empresa, pois não vê oportunidades de crescimento na empresa atual.

Pode surgir a dúvida: A feature RoleStagnation não atende a mesma informação? Não, pois RoleStagnation mede o mesmo cargo em relação ao tempo na empresa e YearsWithoutPromotion mede o tempo sem promoção em relação ao tempo da empresa.  Um funcionário pode ter mudado de cargo (RoleStagnation baixo) mas para um cargo do mesmo nível (YearsWithoutPromotion alto), são informações complementares, não redundantes.

- valor alto = estagnado na carreira, possível frustração e aberto a mudanças fora da empresa.
- valor baixo = promovido recentemente, mais engajado com o trabalho atual.

**Cálculo:** YearsSinceLastPromotion / (YearsAtCompany + 1)

### **Feature 6: AgeGroup**

**Justificativa**: Transformar a variável contínua idade em categorias (Jovem, Adulto, Senior). Pensando no mercado de trabalho real temos algumas hipóteses que pode fazer muito sentido com o nosso contexto:

Jovem com menos de 30 anos:
- Início de carreira, buscando crescimento rápido
- Menos "preso" à empresa (menos responsabilidades financeiras)
- Mais aberto a trocar de emprego por oportunidades ainda melhores
- Tem maior mobilidade

Adulto entre 31 e 45 anos:
- Meio de carreira, buscando estabilidade
- Geralmente tem família, casa, responsabilidade financeira, filhos
- Trocar de emprego pode ser mais arriscado mas é possível que aconteça
- Comportamento moderado

Senior com mais de 45 anos:
- Carreira consolidada
- É mais difícil de se realocar no mercado de trabalho
- Valorizza estabilidade e benefícios
- Tende a buscar maior estabilidade

Jovens tendem a ter mais mobilidade enquanto seniores buscam por estabilidade.



## Features de Compensação

### **Feature 7: IncomePerLevel**

**Justificativa**: Mede a comparação entre o salário com o nível hierárquico do funcionário. Se o salário for baixo para o seu nível, o funcionário pode sentir-se desvalorizado e tem mais risco de entrar em attrition, pois a expectativa é que o salário esteja de acordo com o nível do funcionário.

**Exemplo:** Ana tem um salário de $3.000e está em JobLevel 1. Bruno tem um salário de $3.000 e está em JobLevel 3. Realizando o os cálculos, temos o seguinte resultado.

| Funcionário | Salário | JobLevel | income_per_level | Interpretação |
|---|---|---|---|---|
|Ana | 3.000 |1|3.000 / 1 = 3.000| Bem paga para nível 1|
|Bruno | 3.000 | 3 | 3.000 / 3 = 1.000 | Mal pago para nível 3|

Bruno, tem mais risco de attrition, pois a expectativa salarial de do nível 3 é muito maior do que $3.000.


**Cálculo:** MonthlyIncome / JobLevel


### **Feature 8: IncomeMedianJob**

**Justificativa**: Mede o salário atual do funcionário em relação a mediana de salário dos pares e parceiros que estão no mesmo cargo. Um funcionário abaixo de 1.0 ganha menos que os pares e um funcionário acima de 1.0 ganha mais que os seus pares. Com isso, funcionários que ganham menos que os seus pares e parceiros de um mesmo cargo, tendem a ficarem insatisfeitos e aberto a mudanças fora da empresa.

**Calculo:** MonthlyIncome / mediana do MonthlyIncome por JobRole

## Feature Risco e Comportamento

### **Feature 9 - RiskOvertimeDistance**

**Justificativa:** Mede a interação do funcionário que faz hora extra e mora longe. Normalmente um funcionário que realiza hora extra e mora longe da sua casa, acumula dois fatores de desgaste simultaneamente. O primeiro fator é o desgaste mental do trabalho em fazer mais horas do que é proposto e também o desgaste físico de sair tarde do trabalho e fazer todo o percurso até sua casa. O desgaste físico pode estar relacionado ao trânsito pesado para voltar, trens, metrôs e onibus superlotados, horário de pico, entre outros fatores.

**Cálculo:** (OverTime_encoded) × DistanceFromHome

**Variável OverTime:** Valor binário Yes e No. Será necessário aplicar valores numéricos para realizar o cálculo. Iremos aplicar Yes = 1 e No = 0, chamando a variável de OverTime_encoded.


### **Feature 10 - RiskWorklifeOvertime**

**Justificativa:** Mede a interação entre o equilíbrio vida-trabalho ruim e hora extra. A feature captura que o risco é alto quando os dois fatores se acumulam, pois hora extra sozinha ou equilíbrio ruim sozinho não geram o mesmo efeito para que o modelo entenda a situação de risco do funcionário. A união entre as duas variáveis é muito boa para entender alguns cenários de risco entre a vida e o trabalho do funcionário.

**Cálculo:** (5 - WorkLifeBalance) × (OverTime_encoded)

**Exemplo**:

|WorkLifeBalance|OverTime|Cálculo|Resultado|Risco|
|---|---|---|---|---|
(Excelente)   | No       | (5-4) × 0 = 1×0  |    0      | Sem risco 
(Ruim)        | Yes      | (5-1) × 1 = 4×1  |    4      | Crítico
(Regular)     | Yes      | (5-2) × 1 = 3×1  |    3      | Alto
(Bom)         | Yes      | (5-3) × 1 = 2×1  |    2      | Médio
(Excelente)   | Yes      | (5-4) × 1 = 1×1  |    1      | Baixo

 Sem essa feature, o modelo veria duas variáveis separadas:
- WorkLifeBalance = 1
- OverTime = Yes

-> O modelo teria que aprender sozinho que a combinação dos dois é mais perigosa do que cada uma de forma isolada

Com essa feature, o modelo recebe direto:
- worklife_overtime_risk = 4

-> A informação de INTERAÇÃO já está pronta e facilita muito o trabalho do modelo

# Feature Enginnering

#### O que é Feature Engineering

- É a transformação de dados brutos em novas variáveis com o objetivo de aumentar a capacidade do modelo em realizar previsões. Para o nosso contexto, precisamos realizar a etapa de Feature Engineering para criarmos features que ajudem na identificação de padrões e que se correlacionem com a nossa target (Attrition), pois até o momento, nenhuma das variáveis contidas no dataset apresentou correlação relevante com a target, conforme evidenciado nas três camadas de análise da EDA gráfica, estatística e correlacional.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Carregar o dataset original
df = pd.read_csv('../data/raw/ibm_hr_analytics_sintetico.csv')

# Remover colunas inúteis (análises em 01_analise_exploratoria)
df = df.drop(columns=['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber'])

print(f"Shape: {df.shape}")
df.head()


Shape: (1000000, 31)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,56,No,Travel_Rarely,590,Research & Development,19,1,Other,4,Female,...,3,1,3,33,2,2,29,11,11,1
1,46,No,Travel_Rarely,1441,Human Resources,5,1,Life Sciences,4,Male,...,3,1,2,37,3,3,37,6,5,16
2,32,Yes,Travel_Frequently,748,Research & Development,2,3,Medical,4,Male,...,3,2,0,26,1,1,16,3,15,12
3,60,No,Travel_Rarely,1311,Human Resources,3,4,Life Sciences,2,Female,...,3,4,2,29,3,2,29,17,6,9
4,25,No,Non-Travel,144,Sales,10,4,Life Sciences,4,Male,...,3,2,1,22,6,4,6,2,6,14


#### Criando variável binária para Attrition

In [2]:
df_raw = df.copy()
df["AttritionFlag"] = (df["Attrition"] == "Yes").astype(int)

In [11]:
# Feature 1: SatisfactionScore
df['SatisfactionScore'] = (df['JobSatisfaction'] + df['EnvironmentSatisfaction'] + df['RelationshipSatisfaction']) / 3

# Analise do resultado
print(f"Mínimo: {df['SatisfactionScore'].min():.2f}")
print(f"Máximo: {df['SatisfactionScore'].max():.2f}")
print(f"Média:  {df['SatisfactionScore'].mean():.2f}")
df[['JobSatisfaction', 'EnvironmentSatisfaction', 'RelationshipSatisfaction', 'SatisfactionScore']].head(10)

# Análise do resultado
# Mínimo: 1.00: funcionário insatisfeito em tudo
# Máximo: 4.00: funcionário satisfeito em tudo
# Média:  2.50: ponto médio exato da escala (1 a 4)


Mínimo: 1.00
Máximo: 4.00
Média:  2.50


,JobSatisfaction,EnvironmentSatisfaction,RelationshipSatisfaction,SatisfactionScore
0,3,4,1,2.666667
1,3,4,1,2.666667
2,2,4,2,2.666667
3,2,2,4,2.666667
4,3,4,2,3.000000
5,1,2,3,2.000000
6,1,1,3,1.666667
7,2,4,2,2.666667
8,2,3,1,2.000000
9,4,3,2,3.000000


In [12]:
# Feature 2: CriticalSatisfactionFlag
df['CriticalSatisfactionFlag'] = (df['SatisfactionScore'] <= 2).astype(int)

# Verificar distribuição
print(df['CriticalSatisfactionFlag'].value_counts())
print(f"\n% críticos: {df['CriticalSatisfactionFlag'].mean() * 100:.2f}%")

# Análise do resultado
# ~31% dos funcionários têm satisfação crítica (score ≤ 2.0), é uma proporção boa
# nem muito restritiva nem muito ampla. O modelo pode ter volume suficiente nos dois grupos para aprender a diferenciar.


CriticalSatisfactionFlag
0    686909
1    313091
Name: count, dtype: int64

% críticos: 31.31%


In [13]:
# Feature 3: RatioCareerCompany
df['RatioCareerCompany'] = df['YearsAtCompany'] / (df['TotalWorkingYears'] + 1)

# Verificar o resultado
print(f"Mínimo: {df['RatioCareerCompany'].min():.2f}")
print(f"Máximo: {df['RatioCareerCompany'].max():.2f}")
print(f"Média:  {df['RatioCareerCompany'].mean():.2f}")
df[['YearsAtCompany', 'TotalWorkingYears', 'RatioCareerCompany']].head(10)

# Análise do resultado

# A feature está conseguindo diferenciar bem os dois perfis (Quem fez a carreira na empresa e quem passou por outras 
# empresas ao longo do tempo)

# Podemos pegar dois exemplos: 

# Linha 1: 37 anos na empresa / 37 de carreira = 0.97
# -> Praticamente fez toda a sua carreira dentro da TechCorp Brasil

# Linha 4: 6 anos na empresa / 22 de carreira = 0.26
# -> Passou por outras empresas

# TODO: Validar o atributo NumCompaniesWorked em relação TotalWorkingYears e Validar os cenários visto acima, para ver se faz sentido. 
# Pode ser uma feature nova, mas que provavelmente terá muita correlação com essa que estamos criando. 


Mínimo: 0.00
Máximo: 0.97
Média:  0.71


,YearsAtCompany,TotalWorkingYears,RatioCareerCompany
0,29,33,0.852941
1,37,37,0.973684
2,16,26,0.592593
3,29,29,0.966667
4,6,22,0.260870
5,18,25,0.692308
6,33,33,0.970588
7,29,29,0.966667
8,3,4,0.600000
9,31,38,0.794872


In [14]:
# Feature 4: RoleStagnation
df['RoleStagnation'] = df['YearsInCurrentRole'] / (df['YearsAtCompany'] + 1)

# Verificar o resultado
print(f"Mínimo: {df['RoleStagnation'].min():.2f}")
print(f"Máximo: {df['RoleStagnation'].max():.2f}")
print(f"Média:  {df['RoleStagnation'].mean():.2f}")
df[['YearsInCurrentRole', 'YearsAtCompany', 'RoleStagnation']].head(10)

# Análise do resultado
# A feature está conseguindo diferenciar bem os funcionários que estão em um mesmo cargo a muito tempo. Quanto mais próximo de 1.0, signifca que o funcionário está a
# mais tempo no mesmo cargo e estagnado.

Mínimo: 0.00
Máximo: 0.95
Média:  0.44


,YearsInCurrentRole,YearsAtCompany,RoleStagnation
0,11,29,0.366667
1,6,37,0.157895
2,3,16,0.176471
3,17,29,0.566667
4,2,6,0.285714
5,17,18,0.894737
6,17,33,0.500000
7,0,29,0.000000
8,3,3,0.750000
9,5,31,0.156250


In [15]:
# Feature 5: YearsWithoutPromotion
df['YearsWithoutPromotion'] = df['YearsSinceLastPromotion'] / (df['YearsAtCompany'] + 1)

# Verificar o resultado
print(f"Mínimo: {df['YearsWithoutPromotion'].min():.2f}")
print(f"Máximo: {df['YearsWithoutPromotion'].max():.2f}")
print(f"Média:  {df['YearsWithoutPromotion'].mean():.2f}")
df[['YearsSinceLastPromotion', 'YearsAtCompany', 'YearsWithoutPromotion']].head(10)

# Análise do resultado
# A feature consegue identificar os perfis e o momento atual do funcionário em relação as expectativas do mercado. Conseguimos encontrar funcionários que estão a muito tempo na empresa sem promoção, 
# mas também, identificando casos em que conseguiu promoções em relação ao total de anos na empresa.

Mínimo: 0.00
Máximo: 0.94
Média:  0.39


,YearsSinceLastPromotion,YearsAtCompany,YearsWithoutPromotion
0,11,29,0.366667
1,5,37,0.131579
2,15,16,0.882353
3,6,29,0.200000
4,6,6,0.857143
5,3,18,0.157895
6,13,33,0.382353
7,13,29,0.433333
8,3,3,0.750000
9,3,31,0.093750


In [27]:
# Feature 6: AgeGroup
df['AgeGroup'] = pd.cut(
    df['Age'],
    bins=[0, 30, 45, 100],
    labels=['Jovem', 'Adulto', 'Senior']
)

# Verificar distribuição
print(df['AgeGroup'].value_counts())
print(f"\nProporção:")
print(df['AgeGroup'].value_counts(normalize=True).round(4) * 100)

df[['Age', 'AgeGroup']].head()

# Análise do resultado
# A feature AgeGroup transforma a idade contínua em faixas de comportamento do mercado de trabalho. A distribuição ficou equilibrada entre os 3 grupos: Senior (40,42%), Adulto (31,95%) e Jovem (27,63%). 
# Isso garante que o modelo terá volume suficiente em cada faixa para aprender padrões de comportamento distintos entre as fases de carreira. 
# Aparentemente, a distribuição está boa, pois nenhuma das categorias ficou com volume muito pequeno, o que pode ser relevante para o modelo conseguir aprender os padrões de cada categoria.

AgeGroup
Senior    404231
Adulto    319486
Jovem     276283
Name: count, dtype: int64

Proporção:
AgeGroup
Senior    40.42
Adulto    31.95
Jovem     27.63
Name: proportion, dtype: float64


,Age,AgeGroup
0,56,Senior
1,46,Senior
2,32,Adulto
3,60,Senior
4,25,Jovem


In [28]:
# Feature 7: IncomePerLevel
df['IncomePerLevel'] = df['MonthlyIncome'] / df['JobLevel']

# Verificar o resultado
print(f"Mínimo: {df['IncomePerLevel'].min():.2f}")
print(f"Máximo: {df['IncomePerLevel'].max():.2f}")
print(f"Média:  {df['IncomePerLevel'].mean():.2f}")
df[['MonthlyIncome', 'JobLevel', 'IncomePerLevel']].head(10)

# A feature IncomePerLevel está medindo muito bem o salário do funcionário em relação ao seu nível (1-5). Com essa variável, podeemos ver se o funcionário está sendo valorizado ou desvalorizado para o seu nível. 
# Em caso de desvalorização, o funcionário tende a entrar em attrition com a empresa. Exemplo da linha 4: O funcionario recebe 1807 estando no nível 4 tendo um resultado de IncomePerLevel = 451.750000, 
# o que é um valor muito abaixo da sua expectativa. Esse funcionário tende a procurar oportunidades melhores fora da empresa.

Mínimo: 200.00
Máximo: 19999.00
Média:  4792.53


,MonthlyIncome,JobLevel,IncomePerLevel
0,12378,3,4126.000000
1,9987,1,9987.000000
2,1276,2,638.000000
3,1807,4,451.750000
4,14635,2,7317.500000
5,11612,5,2322.400000
6,18291,4,4572.750000
7,7871,3,2623.666667
8,16812,3,5604.000000
9,11938,2,5969.000000


In [30]:
# Feature 8: IncomeMedianJob
# Calcula a mediana de salário por cargo e compara com o salário do funcionário
median_by_role = df.groupby('JobRole')['MonthlyIncome'].transform('median')
df['IncomeMedianJob'] = df['MonthlyIncome'] / median_by_role

# Verificar o resultado
print(f"Mínimo: {df['IncomeMedianJob'].min():.2f}")
print(f"Máximo: {df['IncomeMedianJob'].max():.2f}")
print(f"Média:  {df['IncomeMedianJob'].mean():.2f}")
df[['JobRole', 'MonthlyIncome', 'IncomeMedianJob']].head(10)

# Os resultados mostram que a feature varia entre 0.09 e 1.91, com média de 1.00. 
# Exemplos:
# Linha 2 (Sales Representative, salário 1.276) com IncomeMedianJob de 0.12, ou seja, ganha apenas 12% da mediana do cargo.
# Linha 6 (mesmo cargo, salário 18.291) tem IncomeMedianJob de 1.74, ganhando 74% acima da mediana. 

# A feature diferencia bem funcionários subvalorizados dentro do mesmo cargo.

Mínimo: 0.09
Máximo: 1.91
Média:  1.00


,JobRole,MonthlyIncome,IncomeMedianJob
0,Research Director,12378,1.175722
1,Laboratory Technician,9987,0.951324
2,Sales Representative,1276,0.121478
3,Research Scientist,1807,0.173001
4,Manager,14635,1.393279
5,Manufacturing Director,11612,1.106959
6,Sales Representative,18291,1.741337
7,Sales Representative,7871,0.749334
8,Manufacturing Director,16812,1.602669
9,Laboratory Technician,11938,1.137169


In [32]:
# Feature 9: RiskOvertimeDistance
# Encoding do OverTime: Yes=1, No=0
df['OverTime_encoded'] = (df['OverTime'] == 'Yes').astype(int)

df['RiskOvertimeDistance'] = df['OverTime_encoded'] * df['DistanceFromHome']

# Verificar o resultado
print(f"Mínimo: {df['RiskOvertimeDistance'].min():.2f}")
print(f"Máximo: {df['RiskOvertimeDistance'].max():.2f}")
print(f"Média:  {df['RiskOvertimeDistance'].mean():.2f}")
df[['OverTime', 'DistanceFromHome', 'RiskOvertimeDistance']].head(10)

# Os resultados variam entre 0.00 e 29.00, com média de 4.20. A feature funciona como um interruptor: funcionários sem hora extra (OverTime=No) recebem valor 0 independente da distância.
# exemplo: 
# Linha 8, distância 29 mas resultado 0. 

# Já quem faz hora extra, o valor reflete a distância, exemplo:
# Linha 0, OverTime=Yes e distância 19 = risco 19. 

# A feature captura bem o acúmulo de desgaste entre hora extra e distância.

Mínimo: 0.00
Máximo: 29.00
Média:  4.20


,OverTime,DistanceFromHome,RiskOvertimeDistance
0,Yes,19,19
1,Yes,5,5
2,No,2,0
3,No,3,0
4,No,10,0
5,No,10,0
6,No,23,0
7,No,24,0
8,No,29,0
9,Yes,2,2


In [33]:
# Feature 10: RiskWorklifeOvertime
df['RiskWorklifeOvertime'] = (5 - df['WorkLifeBalance']) * df['OverTime_encoded']

# Verificar o resultado
print(f"Mínimo: {df['RiskWorklifeOvertime'].min():.2f}")
print(f"Máximo: {df['RiskWorklifeOvertime'].max():.2f}")
print(f"Média:  {df['RiskWorklifeOvertime'].mean():.2f}")
df[['WorkLifeBalance', 'OverTime', 'RiskWorklifeOvertime']].head(10)

# Os resultados variam entre 0.00 e 4.00, com média de 0.70. A feature captura bem os cenários extremos: 
# Linha 9 (WorkLifeBalance=1 e OverTime=Yes) recebe valor 4 (risco crítico — equilíbrio péssimo com hora extra). 

# Já funcionários sem hora extra recebem sempre 0, independente do WorkLifeBalance:
# Linha 2, equilíbrio ruim mas sem hora extra = 0). 

# A feature diferencia bem o acúmulo de sobrecarga.

Mínimo: 0.00
Máximo: 4.00
Média:  0.70


,WorkLifeBalance,OverTime,RiskWorklifeOvertime
0,2,Yes,3
1,3,Yes,2
2,1,No,0
3,2,No,0
4,4,No,0
5,1,No,0
6,3,No,0
7,1,No,0
8,3,No,0
9,1,Yes,4


In [37]:
df.shape

(1000000, 43)

In [38]:
print(f"Novas colunas: {df.shape[1] - 26} features criadas")

Novas colunas: 17 features criadas


In [40]:
# Salvando o dataset com as features criadas
df.to_csv('../data/processed/ibm_hr_analytics_feature_engineering.csv', index=False)

As features também foram atualizadas no arquivo 01_metadados. Por lá, conseguimos salvar as features com as suas respectivas descrições.